In [1]:
import vtk

# Load the dataset
reader = vtk.vtkXMLImageDataReader()
reader.SetFileName("Isabel_2D.vti")  # Replace with actual file path
reader.Update()

data = reader.GetOutput()

# Get dimensions
dims = data.GetDimensions()
print("Dimensions:", dims)

# Number of points and cells
num_points = data.GetNumberOfPoints()
num_cells = data.GetNumberOfCells()
print("Number of points:", num_points)
print("Number of cells:", num_cells)

# Pressure data
pressure_array = data.GetPointData().GetArray("Pressure")

# Range of Pressure
min_pressure, max_pressure = pressure_array.GetRange()
print("Pressure range:", (min_pressure, max_pressure))

# Average Pressure
total_pressure = sum(pressure_array.GetValue(i) for i in range(pressure_array.GetNumberOfTuples()))
avg_pressure = total_pressure / pressure_array.GetNumberOfTuples()
print("Average Pressure:", avg_pressure)

# --- Cell Info (parameterized by cell_id) ---
cell_id = 0  # You can make this dynamic
cell = data.GetCell(cell_id)

# Indices of vertices
point_ids = cell.GetPointIds()
print("Vertex indices:", [point_ids.GetId(i) for i in range(point_ids.GetNumberOfIds())])

# Coordinates of vertices
points = data.GetPoints()
coords = [points.GetPoint(point_ids.GetId(i)) for i in range(4)]
print("Vertex coordinates:", coords)

# Center of cell
center = tuple(sum(coord[i] for coord in coords)/4 for i in range(3))
print("Cell center coordinate:", center)

# Pressure values at vertices
pressures = [pressure_array.GetValue(point_ids.GetId(i)) for i in range(4)]
print("Pressure at vertices:", pressures)

# Average Pressure at cell center
avg_cell_pressure = sum(pressures) / 4
print("Average Pressure at cell center:", avg_cell_pressure)

Dimensions: (250, 250, 1)
Number of points: 62500
Number of cells: 62001
Pressure range: (-1434.8590087890625, 630.5694580078125)
Average Pressure: 240.77722069091325
Vertex indices: [0, 1, 250, 251]
Vertex coordinates: [(0.0, 0.0, 25.0), (1.0, 0.0, 25.0), (0.0, 1.0, 25.0), (1.0, 1.0, 25.0)]
Cell center coordinate: (0.5, 0.5, 25.0)
Pressure at vertices: [477.527587890625, 474.79827880859375, 478.0115661621094, 467.60699462890625]
Average Pressure at cell center: 474.4861068725586


In [2]:
## This program shows how to load a VTK Image data with extension *.vti and then
## how to access the cell data, extract one cell from the list of cells,
## access data values at each cell corner and then store the cell into a 
## vtkpolydata file format.
#################################################################################

## Import VTK
from vtk import *

## Load data
#######################################
reader = vtkXMLImageDataReader()
reader.SetFileName('Isabel_2D.vti')
reader.Update()
data = reader.GetOutput()

## Query how many cells the dataset has
#######################################
numCells = data.GetNumberOfCells()

## Get a single cell from the list of cells
###########################################
cell = data.GetCell(0) ## cell index = 0

## Query the 4 corner points of the cell
#########################################
pid1 = cell.GetPointId(0)
pid2 = cell.GetPointId(1)
pid3 = cell.GetPointId(3)
pid4 = cell.GetPointId(2)

## Print the 1D indices of the corner points
############################################
print('1D indices of the cell corner points:')
print(pid1,pid2,pid3,pid4) ## in counter-clockwise order


## Get values at each vertex
## First Get the array
dataArr = data.GetPointData().GetArray('Pressure')
val1 = dataArr.GetTuple1(pid1)
val2 = dataArr.GetTuple1(pid2)
val3 = dataArr.GetTuple1(pid3)
val4 = dataArr.GetTuple1(pid4)
#print(val1,val2,val3,val4)

## Print the locations (3D coordinates) of the points
#######################################################
print('Point locations of cell corners in counter clockwise order and their data values:')
print(data.GetPoint(pid1),val1)
print(data.GetPoint(pid2),val2)
print(data.GetPoint(pid3),val3)
print(data.GetPoint(pid4),val4)


## Extract and store one cell
#############################
points = vtkPoints()
points.InsertNextPoint(data.GetPoint(pid1))
points.InsertNextPoint(data.GetPoint(pid2))
points.InsertNextPoint(data.GetPoint(pid3))
points.InsertNextPoint(data.GetPoint(pid4))

## The Data Array for holding data values
#########################################
dataArray = vtkFloatArray()
dataArray.SetName('Pressure')
dataArray.InsertNextTuple1(val1)
dataArray.InsertNextTuple1(val2)
dataArray.InsertNextTuple1(val3)
dataArray.InsertNextTuple1(val4)

## Create the cell, which is a polyline
#########################################
polyLine = vtkPolyLine()
polyLine.GetPointIds().SetNumberOfIds(5)    
polyLine.GetPointIds().SetId(0, 0)
polyLine.GetPointIds().SetId(1, 1)
polyLine.GetPointIds().SetId(2, 2)
polyLine.GetPointIds().SetId(3, 3)
polyLine.GetPointIds().SetId(4, 0)

### Create a polydata
######################
pdata = vtkPolyData()

### Create a cell array to store the polyline
#############################################
cells = vtkCellArray()
cells.InsertNextCell(polyLine)

### Add points and cells to polydata
####################################
pdata.SetPoints(points)
pdata.SetLines(cells)
pdata.GetPointData().AddArray(dataArray)


### Store the polydata into a vtkpolydata file with extension .vtp
####################################################################
writer = vtkXMLPolyDataWriter()
writer.SetInputData(pdata)
writer.SetFileName('onecell.vtp')
writer.Write()

1D indices of the cell corner points:
0 1 251 250
Point locations of cell corners in counter clockwise order and their data values:
(0.0, 0.0, 25.0) 477.527587890625
(1.0, 0.0, 25.0) 474.79827880859375
(1.0, 1.0, 25.0) 467.60699462890625
(0.0, 1.0, 25.0) 478.0115661621094


1

In [3]:
## How to visualize a 2D uniform grid colored with the array attribute.
#############################################################################

## Import VTK
from vtk import *
#################################


## Load data
#######################################
reader = vtkXMLImageDataReader()
reader.SetFileName('Isabel_2D.vti')
reader.Update()
data = reader.GetOutput()

## create a surface representation from 2D uniform grid data
surface = vtkGeometryFilter()
surface.SetInputData(data)
surface.Update()

## Output of geometry filter is a vtkpolydata
pdata = surface.GetOutput()
range = pdata.GetPointData().GetArray('Pressure').GetRange()


# create the scalar_bar
##########################
lut = vtkLookupTable()
lut.Build()
scalar_bar = vtkScalarBarActor()
scalar_bar.SetLookupTable(lut)
scalar_bar.GetTitleTextProperty().SetColor(0,0,0)
scalar_bar.GetProperty().SetColor(0,0,0)
scalar_bar.SetTitle("Pressure")
scalar_bar.SetNumberOfLabels(6)
scalar_bar.SetMaximumWidthInPixels(100)
scalar_bar.SetMaximumHeightInPixels(300)


### Setup mapper and actor
##########################
mapper = vtkPolyDataMapper()
mapper.SetInputData(pdata)
mapper.SetScalarRange(range)
mapper.SetLookupTable(lut)
actor = vtkActor()
actor.SetMapper(mapper)

### Axes actor
###############
axes = vtkAxesActor()
axes.SetTotalLength(50, 50, 50)
axes.AxisLabelsOff()


### Setup render window, renderer, and interactor
##################################################
renderer = vtkRenderer()
renderer.SetBackground(1,1,1)
render_window = vtkRenderWindow()
render_window.SetSize(1400,1000)
render_window.AddRenderer(renderer)
render_windowInteractor = vtkRenderWindowInteractor()
render_windowInteractor.SetRenderWindow(render_window)
render_windowInteractor.SetInteractorStyle(vtkInteractorStyleTrackballCamera())
renderer.AddActor(actor)
renderer.AddActor(axes)

## show the scalar bar
scalar_bar_widget = vtkScalarBarWidget()
scalar_bar_widget.SetInteractor(render_windowInteractor)
scalar_bar_widget.SetScalarBarActor(scalar_bar)
scalar_bar_widget.On()


### Finally render the object
#############################
render_window.Render()
render_windowInteractor.Start()

In [4]:
del range

In [5]:
print(type(range))

<class 'type'>


In [6]:
import os
import vtk

def interpolate(p1, p2, val1, val2, isovalue):
    t = (isovalue - val1) / (val2 - val1)
    return [p1[i] + t * (p2[i] - p1[i]) for i in range(3)]

def extract_isocontour(filename, isovalue, output_filename):

    reader = vtk.vtkXMLImageDataReader()
    reader.SetFileName(filename)
    reader.Update()
    image_data = reader.GetOutput()

    scalars = image_data.GetPointData().GetScalars()

    dims = image_data.GetDimensions()
    scalars = image_data.GetPointData().GetScalars()

    points = vtk.vtkPoints()
    lines = vtk.vtkCellArray()
    point_id = 0

    for i in range(dims[0] - 1):
        for j in range(dims[1] - 1):
            pt_ids = [
                j * dims[0] + i,
                j * dims[0] + i + 1,
                (j + 1) * dims[0] + i,
                (j + 1) * dims[0] + i + 1
            ]

            coords = [
                image_data.GetPoint(pt_ids[0]),
                image_data.GetPoint(pt_ids[1]),
                image_data.GetPoint(pt_ids[3]),
                image_data.GetPoint(pt_ids[2])
            ]
            values = [
                scalars.GetTuple1(pt_ids[0]),
                scalars.GetTuple1(pt_ids[1]),
                scalars.GetTuple1(pt_ids[3]),
                scalars.GetTuple1(pt_ids[2])
            ]

            edge_pairs = [(0,1), (1,2), (2,3), (3,0)]
            edge_points = []

            for idx1, idx2 in edge_pairs:
                val1, val2 = values[idx1], values[idx2]
                if (val1 - isovalue) * (val2 - isovalue) < 0:
                    pt = interpolate(coords[idx1], coords[idx2], val1, val2, isovalue)
                    edge_points.append(pt)

            if len(edge_points) == 2:
                id1 = points.InsertNextPoint(edge_points[0])
                id2 = points.InsertNextPoint(edge_points[1])
                line = vtk.vtkLine()
                line.GetPointIds().SetId(0, id1)
                line.GetPointIds().SetId(1, id2)
                lines.InsertNextCell(line)

    polydata = vtk.vtkPolyData()
    polydata.SetPoints(points)
    polydata.SetLines(lines)

    writer = vtk.vtkXMLPolyDataWriter()
    writer.SetFileName(output_filename)
    writer.SetInputData(polydata)
    writer.Write()

    # Optional live visualization window
    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputData(polydata)
    
    actor = vtk.vtkActor()
    actor.SetMapper(mapper)
    actor.GetProperty().SetColor(1, 0, 0)  # red contour lines
    
    renderer = vtk.vtkRenderer()
    renderer.AddActor(actor)
    renderer.SetBackground(1.0, 1.0, 1.0)  # white background
    
    render_window = vtk.vtkRenderWindow()
    render_window.AddRenderer(renderer)
    render_window.SetSize(800, 800)
    
    interactor = vtk.vtkRenderWindowInteractor()
    interactor.SetRenderWindow(render_window)
    
    render_window.Render()
    interactor.Start()


if __name__ == "__main__":
    import sys
    if len(sys.argv) == 4:
        input_file = sys.argv[1]
        isovalue = float(sys.argv[2])
        output_file = sys.argv[3]
        
extract_isocontour('Isabel_2D.vti',430, 'output.vtp')


In [7]:
import sys
import vtk

def render_volume(input_file, enable_shading):
    # Read VTI
    reader = vtk.vtkXMLImageDataReader()
    reader.SetFileName(input_file)
    reader.Update()

    # Mapper
    volume_mapper = vtk.vtkSmartVolumeMapper()
    volume_mapper.SetInputConnection(reader.GetOutputPort())

    # Opacity Transfer Function (example)
    opacity = vtk.vtkPiecewiseFunction()
    opacity.AddPoint(-1500, 0.0)
    opacity.AddPoint(0, 0.05)
    opacity.AddPoint(400, 0.3)
    opacity.AddPoint(800, 0.6)

    # Color Transfer Function (example)
    color = vtk.vtkColorTransferFunction()
    color.AddRGBPoint(-1500, 0.0, 0.0, 0.0)
    color.AddRGBPoint(-500, 0.0, 0.0, 1.0)
    color.AddRGBPoint(0, 0.0, 1.0, 1.0)
    color.AddRGBPoint(500, 1.0, 1.0, 0.0)
    color.AddRGBPoint(1000, 1.0, 0.0, 0.0)

    # Volume Property
    volume_property = vtk.vtkVolumeProperty()
    volume_property.SetColor(color)
    volume_property.SetScalarOpacity(opacity)
    volume_property.SetInterpolationTypeToLinear()
    volume_property.SetShade(enable_shading)
    if enable_shading:
        volume_property.SetAmbient(0.5)
        volume_property.SetDiffuse(0.5)
        volume_property.SetSpecular(0.5)

    # Volume
    volume = vtk.vtkVolume()
    volume.SetMapper(volume_mapper)
    volume.SetProperty(volume_property)

    # Outline
    outline = vtk.vtkOutlineFilter()
    outline.SetInputConnection(reader.GetOutputPort())
    outline_mapper = vtk.vtkPolyDataMapper()
    outline_mapper.SetInputConnection(outline.GetOutputPort())
    outline_actor = vtk.vtkActor()
    outline_actor.SetMapper(outline_mapper)

    # Renderer
    renderer = vtk.vtkRenderer()
    renderer.AddVolume(volume)
    renderer.AddActor(outline_actor)
    renderer.SetBackground(1, 1, 1)

    # Render window
    render_window = vtk.vtkRenderWindow()
    render_window.AddRenderer(renderer)
    render_window.SetSize(1000, 1000)

    # Interactor
    interactor = vtk.vtkRenderWindowInteractor()
    interactor.SetRenderWindow(render_window)

    # Launch
    render_window.Render()
    interactor.Start()

render_volume("Isabel_3D.vti", True)